# SPCS LLM Streaming Benchmark

A generalized benchmark for any SPCS-hosted LLM service that exposes an OpenAI-compatible `/v1/chat/completions` endpoint.

**Features:**
- Works with any SPCS model (Llama, Mistral, Gemma, etc.)
- Uses real-world prompts from the [LM Arena](https://huggingface.co/datasets/lmarena-ai/arena-human-preference-55k) dataset
- Configurable concurrency sweep (1 to N concurrent clients)
- Measures TTFT, end-to-end latency, throughput (tokens/s), and requests/s
- Exports results to CSV for further analysis

In [ ]:
%pip install -q openai pandas datasets matplotlib

## Configuration

Set your SPCS endpoint, model name, and authentication token below.

In [ ]:
# ── SPCS Endpoint ─────────────────────────────────────────────────────────────
SPCS_BASE_URL = "https://<endpoint_name>/v1"  # CHANGE ME
SPCS_MODEL = "<model_name>"                                       # CHANGE ME
PAT = "<pat_token>"                               # CHANGE ME

# ── Generation parameters ─────────────────────────────────────────────────────
MAX_COMPLETION_TOKENS = 150
TEMPERATURE = 0.9
TOP_P = 1.0

# ── Prompt dataset settings ───────────────────────────────────────────────────
NUM_PROMPTS = 200          # how many prompts to sample from the arena dataset
SYSTEM_PROMPT = "You are a helpful assistant."  # set to None to omit

# ── Benchmark sweep ───────────────────────────────────────────────────────────
configs = [
    {"clients": 1,   "duration_seconds": 180},
    {"clients": 5,   "duration_seconds": 180},
    {"clients": 10,  "duration_seconds": 180},
    {"clients": 20,  "duration_seconds": 180},
    {"clients": 30,  "duration_seconds": 180},
    {"clients": 50,  "duration_seconds": 180},
    {"clients": 100, "duration_seconds": 180},
]

print(f"Endpoint : {SPCS_BASE_URL}")
print(f"Model    : {SPCS_MODEL}")

## Load prompts from LM Arena dataset

We sample real user prompts from [lmarena-ai/arena-human-preference-55k](https://huggingface.co/datasets/lmarena-ai/arena-human-preference-55k) so the benchmark reflects realistic workloads with diverse prompt lengths and topics.

In [ ]:
from datasets import load_dataset
import random

ds = load_dataset("lmarena-ai/arena-human-preference-55k", split="train")

raw_prompts = [row["prompt"] for row in ds if row["prompt"] and len(row["prompt"].strip()) > 10]
random.seed(42)
sampled_prompts = random.sample(raw_prompts, min(NUM_PROMPTS, len(raw_prompts)))

MESSAGES = []
for prompt_text in sampled_prompts:
    msgs = []
    if SYSTEM_PROMPT:
        msgs.append({"role": "system", "content": SYSTEM_PROMPT})
    msgs.append({"role": "user", "content": prompt_text.strip()})
    MESSAGES.append(msgs)

prompt_lengths = [len(m[-1]["content"]) for m in MESSAGES]
print(f"Loaded {len(MESSAGES)} prompts from LM Arena dataset")
print(f"Prompt length — min: {min(prompt_lengths)}, median: {sorted(prompt_lengths)[len(prompt_lengths)//2]}, max: {max(prompt_lengths)}")

## Initialize OpenAI client

In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key="unused",
    base_url=SPCS_BASE_URL,
    default_headers={"Authorization": f'Snowflake Token="{PAT}"'},
    timeout=300.0,
)

print("OpenAI client initialized.")

## Streaming request helper

In [ ]:
import time


def streaming_request(messages):
    """Fire a single streaming chat completion and return timing metrics."""
    t0 = time.monotonic()
    try:
        stream = client.chat.completions.create(
            model=SPCS_MODEL,
            messages=messages,
            max_completion_tokens=MAX_COMPLETION_TOKENS,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            stream=True,
        )

        first_token_time = None
        last_token_time = None
        tokens = []

        for chunk in stream:
            now = time.monotonic()
            delta = chunk.choices[0].delta
            content = getattr(delta, "content", None) or getattr(delta, "reasoning", None) or ""
            if content:
                if first_token_time is None:
                    first_token_time = now
                last_token_time = now
                tokens.append(content)

        text = "".join(tokens)
        token_count = len(tokens)

        if first_token_time is None:
            return {
                "ttft_ms": None, "latency_ms": (time.monotonic() - t0) * 1000,
                "tps": 0, "token_count": 0, "text": text, "status": "success",
            }

        ttft_ms = (first_token_time - t0) * 1000
        latency_ms = (last_token_time - t0) * 1000
        gen_secs = last_token_time - first_token_time
        tps = token_count / gen_secs if gen_secs > 0 else 0.0

        return {
            "ttft_ms": ttft_ms, "latency_ms": latency_ms,
            "tps": tps, "token_count": token_count, "text": text, "status": "success",
        }

    except Exception as e:
        return {
            "ttft_ms": None, "latency_ms": (time.monotonic() - t0) * 1000,
            "tps": 0, "token_count": 0, "text": "",
            "status": "error", "error": str(e)[:300],
        }


print("Streaming request helper loaded.")

## Smoke test

Send a single request to verify connectivity and print basic metrics.

In [ ]:
res = streaming_request(MESSAGES[0])

if res["ttft_ms"] is not None:
    print(f"TTFT    : {res['ttft_ms']:.1f} ms")
    print(f"Latency : {res['latency_ms']:.1f} ms")
    print(f"Tokens  : {res['token_count']}")
    print(f"TPS     : {res['tps']:.1f}")
    print(f"Preview : {res['text'][:200]}...")
else:
    print(f"No tokens received. Status: {res['status']}")
    if "error" in res:
        print(f"Error: {res['error']}")

## Benchmark runner

Fires concurrent streaming requests in batches for a fixed duration at each concurrency level.  
Per-iteration throughput is measured as total tokens generated across all clients / wall-clock time.

In [ ]:
import statistics
import concurrent.futures


def _percentile(sorted_arr, q):
    if not sorted_arr:
        return 0
    idx = min(int(q * (len(sorted_arr) - 1)), len(sorted_arr) - 1)
    return sorted_arr[idx]


def run_benchmark(num_clients, duration_seconds=60):
    """Run streaming benchmark at a given concurrency for a fixed wall-clock duration."""
    end_time = time.time() + duration_seconds
    iterations = []
    request_idx = 0

    with concurrent.futures.ThreadPoolExecutor(max_workers=num_clients) as pool:
        while time.time() < end_time:
            iter_start = time.monotonic()
            futures = []
            for _ in range(num_clients):
                msgs = MESSAGES[request_idx % len(MESSAGES)]
                futures.append(pool.submit(streaming_request, msgs))
                request_idx += 1

            iter_results = [f.result() for f in concurrent.futures.as_completed(futures)]
            iter_elapsed = time.monotonic() - iter_start

            successes = [r for r in iter_results if r["status"] == "success" and r["token_count"] > 0]
            iter_tokens = sum(r["token_count"] for r in successes)
            iter_tps = iter_tokens / iter_elapsed if iter_elapsed > 0 else 0

            iterations.append({
                "results": iter_results,
                "elapsed_s": iter_elapsed,
                "tokens": iter_tokens,
                "tps": iter_tps,
                "successes": len(successes),
                "failures": len(iter_results) - len(successes),
            })

    if not iterations:
        return {}

    all_results = [r for it in iterations for r in it["results"]]
    successes = [r for r in all_results if r["status"] == "success" and r["token_count"] > 0]
    errors = [r for r in all_results if r["status"] == "error"]

    ttfts = sorted(r["ttft_ms"] for r in successes if r["ttft_ms"] is not None)
    latency_s = sorted(r["latency_ms"] for r in successes)
    iter_tps_vals = [it["tps"] for it in iterations if it["tps"] > 0]
    iter_rps_vals = [
        sum(1 for r in it["results"] if r["status"] == "success") / it["elapsed_s"]
        for it in iterations if it["elapsed_s"] > 0
    ]

    return {
        "clients": num_clients,
        "iterations": len(iterations),
        "total_requests": len(all_results),
        "successful": len(successes),
        "errors": len(errors),
        "success_rate_pct": len(successes) / len(all_results) * 100 if all_results else 0,
        "total_tokens": sum(it["tokens"] for it in iterations),
        "tps_median": statistics.median(iter_tps_vals) if iter_tps_vals else 0,
        "tps_avg": statistics.fmean(iter_tps_vals) if iter_tps_vals else 0,
        "tps_p95": _percentile(sorted(iter_tps_vals), 0.95),
        "tps_min": min(iter_tps_vals) if iter_tps_vals else 0,
        "rps_median": statistics.median(iter_rps_vals) if iter_rps_vals else 0,
        "ttft_median_ms": statistics.median(ttfts) if ttfts else 0,
        "ttft_p95_ms": _percentile(ttfts, 0.95),
        "ttft_p99_ms": _percentile(ttfts, 0.99),
        "latency_median_ms": statistics.median(latency_s) if latency_s else 0,
        "latency_p95_ms": _percentile(latency_s, 0.95),
        "latency_p99_ms": _percentile(latency_s, 0.99),
    }


print("Benchmark runner loaded.")

## Run concurrency sweep

In [ ]:
results = []

for cfg in configs:
    c, d = cfg["clients"], cfg["duration_seconds"]
    print(f"\n{'='*70}")
    print(f"Streaming: {c} concurrent clients for {d}s ...", end="", flush=True)
    res = run_benchmark(c, d)
    results.append(res)

    print(f" done  ({res['iterations']} iters, {res['total_requests']} requests)")
    print(f"  Success : {res['successful']}/{res['total_requests']} ({res['success_rate_pct']:.1f}%)  errors={res['errors']}")
    print(f"  TPS     : median={res['tps_median']:.1f}  avg={res['tps_avg']:.1f}  min={res['tps_min']:.1f}  p95={res['tps_p95']:.1f}")
    print(f"  TTFT    : median={res['ttft_median_ms']:.1f}ms  p95={res['ttft_p95_ms']:.1f}ms  p99={res['ttft_p99_ms']:.1f}ms")
    print(f"  Latency : median={res['latency_median_ms']:.1f}ms  p95={res['latency_p95_ms']:.1f}ms  p99={res['latency_p99_ms']:.1f}ms")

print(f"\n{'='*70}")
print("Benchmark sweep complete.")

## Results summary & export

In [ ]:
import pandas as pd
from datetime import datetime

df = pd.DataFrame(results)

display_cols = [
    "clients", "total_requests", "successful", "errors", "success_rate_pct",
    "tps_median", "tps_avg", "tps_p95",
    "ttft_median_ms", "ttft_p95_ms", "ttft_p99_ms",
    "latency_median_ms", "latency_p95_ms", "latency_p99_ms",
    "rps_median", "total_tokens",
]
df_display = df[[c for c in display_cols if c in df.columns]]

print(df_display.to_string(index=False))

model_tag = SPCS_MODEL.lower().replace("/", "_").replace(" ", "_")
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
fname = f"llm_benchmark_{model_tag}_{ts}.csv"
df.to_csv(fname, index=False)
print(f"\nExported to {fname}")

## Visualize throughput and latency vs. concurrency

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Throughput (tokens/s)
axes[0].plot(df["clients"], df["tps_median"], "o-", label="median")
axes[0].plot(df["clients"], df["tps_avg"], "s--", label="avg")
axes[0].set_xlabel("Concurrent Clients")
axes[0].set_ylabel("Tokens / sec")
axes[0].set_title(f"Throughput — {SPCS_MODEL}")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# TTFT
axes[1].plot(df["clients"], df["ttft_median_ms"], "o-", label="median")
axes[1].plot(df["clients"], df["ttft_p95_ms"], "s--", label="p95")
axes[1].plot(df["clients"], df["ttft_p99_ms"], "^:", label="p99")
axes[1].set_xlabel("Concurrent Clients")
axes[1].set_ylabel("TTFT (ms)")
axes[1].set_title(f"Time to First Token — {SPCS_MODEL}")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# End-to-end latency
axes[2].plot(df["clients"], df["latency_median_ms"], "o-", label="median")
axes[2].plot(df["clients"], df["latency_p95_ms"], "s--", label="p95")
axes[2].plot(df["clients"], df["latency_p99_ms"], "^:", label="p99")
axes[2].set_xlabel("Concurrent Clients")
axes[2].set_ylabel("Latency (ms)")
axes[2].set_title(f"End-to-End Latency — {SPCS_MODEL}")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"llm_benchmark_{model_tag}_{ts}.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Chart saved to llm_benchmark_{model_tag}_{ts}.png")